# ConductorNet — A Custom-Built CNN From Scratch

This notebook builds an original convolutional neural network **entirely
from scratch** — no pretrained ImageNet weights, no copying an existing
named architecture's code — to (1) learn exactly how modern efficient CNNs
are put together, function by function, and (2) honestly test whether a
custom-built model can compete with transfer-learned industry-standard
architectures on this specific task.

## Important expectation-setting, upfront

**A from-scratch model is very unlikely to beat EfficientNet-B0 here, and
that's expected, not a failure.** This isn't a coding limitation — it's a
fundamental, well-known property of deep learning: EfficientNet-B0
achieved 97.5% because it started from weights already trained on 1.2
million ImageNet photos, and only had to *adapt* that existing visual
knowledge to our 827 classes. ConductorNet starts from random noise and
has to learn 'what is an edge, what is a color, what is a shape' from
scratch, using only ~3 images per class. That's an enormously harder
problem, and no amount of architecture cleverness fully closes that gap
at this data scale.

**What 'custom' honestly means here:** this combines three well-established,
published CNN building blocks — depthwise separable convolutions
(MobileNet family), squeeze-and-excitation channel attention (SENet),
and residual connections (ResNet) — into an original network configuration
designed specifically for this project. It does not invent new mathematical
primitives; it's a genuinely original *arrangement* of proven ideas, which
is how essentially all modern architectures are actually designed.

**This notebook reuses the exact same leakage-safe data split, training
loop (with early stopping), and evaluation metrics as the previous
comparison notebooks — so results are directly comparable.**

**Prior results, for comparison (from `all4_categories_per_item_comparison.ipynb`):**

| Model | Test Acc | Size | Params |
|---|---|---|---|
| EfficientNet-B0 (pretrained) | 97.5% | 19.6 MB | 5.07M |
| MobileNetV2 (pretrained) | 96.9% | 12.8 MB | 3.28M |
| CLIP ViT-B/32 (frozen) | 95.3% | 577.2 MB | 151.3M |

**Dataset path:**
```
/kaggle/input/datasets/datascientist97/locomotive-collection-images/conductor_dataset
```

## 0. Setup

In [ ]:
import os
import copy
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    top_k_accuracy_score, roc_auc_score, classification_report
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

FORCE_CPU = False
DEVICE = torch.device('cpu') if FORCE_CPU else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

if torch.cuda.is_available() and not FORCE_CPU:
    try:
        t = torch.randn(2, 2).cuda(); _ = t @ t
        print('✅ GPU tensor op succeeded.')
    except Exception as e:
        print('❌ GPU broken — set FORCE_CPU = True, or switch Accelerator type in Kaggle settings.')
        print('Error:', e)

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/datascientist97/locomotive-collection-images/conductor_dataset"
CATEGORIES = ["Locomotives", "Passenger Train Cars", "Freight Cars", "Automobiles"]

IMG_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 40          # from-scratch models typically need more epochs than fine-tuning
EARLY_STOPPING_PATIENCE = 8
LEARNING_RATE = 3e-4     # slightly lower than the transfer-learning runs — training from
                          # scratch with random init is less stable at high learning rates
WIDTH_MULTIPLIER = 1.0   # scales the whole network's channel counts up/down — try 0.5
                          # for a smaller/faster model, or 1.5 for a bigger one

## 1. Data — Same Leakage-Safe Split as Before

Identical logic to the previous notebooks: each item's one real photo is
held out for testing; synthetic augmented variants split between train
and validation.

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
MIN_TRAIN_IMAGES = 2

records = []
for category in CATEGORIES:
    cat_path = os.path.join(DATASET_ROOT, category)
    if not os.path.isdir(cat_path):
        continue
    for products_id in sorted(os.listdir(cat_path)):
        folder = os.path.join(cat_path, products_id)
        if not os.path.isdir(folder):
            continue
        real_files, aug_files = [], []
        for fname in sorted(os.listdir(folder)):
            ext = os.path.splitext(fname)[1].lower()
            if ext not in VALID_EXTS:
                continue
            (aug_files if '_aug_' in fname else real_files).append(fname)
        if not real_files or len(aug_files) < MIN_TRAIN_IMAGES:
            continue
        test_file = real_files[0]
        records.append({'category': category, 'products_id': products_id,
                         'path': os.path.join(folder, test_file), 'split': 'test'})
        random.shuffle(aug_files)
        records.append({'category': category, 'products_id': products_id,
                         'path': os.path.join(folder, aug_files[-1]), 'split': 'val'})
        for f in aug_files[:-1]:
            records.append({'category': category, 'products_id': products_id,
                             'path': os.path.join(folder, f), 'split': 'train'})

split_df = pd.DataFrame(records)
NUM_CLASSES = split_df['products_id'].nunique()
CLASS_NAMES = sorted(split_df['products_id'].unique())
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df = split_df[split_df['split'] == 'test'].reset_index(drop=True)

print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # kept for consistent input scaling, even though we're
IMAGENET_STD = [0.229, 0.224, 0.225]   # not using ImageNet weights — any fixed, sane normalization works

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(4),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CatalogDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        return self.transform(img), class_to_idx[row['products_id']]

train_loader = DataLoader(CatalogDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(CatalogDataset(val_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(CatalogDataset(test_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 2. Building ConductorNet, Step by Step

Each building block below is a separate cell with detailed comments —
read through in order, since each one builds on the previous.

### 2.1 — ConvBlock: the basic unit of almost every CNN

Every modern CNN is built from stacks of this same three-step pattern:
**convolve → normalize → activate**. Understanding this one block is 80%
of understanding how CNNs work at all.

In [ ]:
class ConvBlock(nn.Module):
    """
    The fundamental building block: Conv2d -> BatchNorm2d -> Activation.

    - Conv2d: slides small learnable filters across the image, each filter
      detecting a specific local pattern (an edge, a color transition, a
      texture). Early layers learn simple patterns; later layers combine
      those into complex ones (wheels, windows, whole locomotive shapes).

    - BatchNorm2d: rescales each channel's activations to have a stable
      mean/variance across each training batch. Without this, deep
      networks are notoriously unstable to train — activations can
      explode or vanish as they pass through many layers. This is
      especially important for a from-scratch model like ours, since we
      don't have pretrained weights to start from a 'good' point already.

    - Activation (SiLU): introduces non-linearity. Without a non-linear
      activation between layers, stacking many conv layers would
      mathematically collapse into being equivalent to just ONE linear
      layer, no matter how many you stack -- the network would be unable
      to learn complex patterns. SiLU (x * sigmoid(x)) is a smoother
      alternative to the classic ReLU, used in EfficientNet too, and
      tends to train slightly better in practice.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, groups=1):
        super().__init__()
        padding = kernel_size // 2  # 'same'-style padding: keeps spatial size unchanged when stride=1
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding,
                               groups=groups, bias=False)  # bias=False: BatchNorm already has its own bias-like term, so a conv bias would be redundant
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

### 2.2 — Depthwise Separable Convolution: the efficiency trick

This is the single biggest idea behind why MobileNet-family networks are
so much smaller than older CNNs for similar accuracy.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """
    A standard convolution mixes SPACE and CHANNELS at the same time in one
    expensive operation. Depthwise separable convolution splits that into
    two cheaper steps done back-to-back:

    1. DEPTHWISE conv: each input channel gets its OWN single filter
       (groups=in_channels means channel i only ever looks at channel i).
       This handles the SPATIAL part (finding patterns within each
       channel) but does NOT mix information between channels at all.

    2. POINTWISE conv (a 1x1 conv): mixes information ACROSS channels at
       each pixel location, but does no spatial pattern-finding at all
       (a 1x1 filter only ever looks at one pixel).

    Doing these two cheap steps back-to-back approximates what one
    expensive standard convolution would do, but with roughly 8-9x fewer
    parameters and computations for a typical 3x3 kernel. This is exactly
    the trick that makes MobileNetV2 12.8 MB instead of the 40-80+ MB of
    older architectures like ResNet with similar depth.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = ConvBlock(in_channels, in_channels, kernel_size=3,
                                     stride=stride, groups=in_channels)
        self.pointwise = ConvBlock(in_channels, out_channels, kernel_size=1, stride=1)

    def forward(self, x):
        x = self.depthwise(x)   # spatial filtering, per-channel
        x = self.pointwise(x)   # channel mixing, per-pixel
        return x

### 2.3 — Squeeze-and-Excitation: teaching the network to pay attention

This is a channel-attention mechanism — it lets the network learn which
of its own feature channels matter most for the current image, and
amplify or suppress them accordingly.

In [ ]:
class SqueezeExcite(nn.Module):
    """
    Not every channel a conv layer produces is equally useful for every
    image. Squeeze-and-Excitation (from the 2018 SENet paper) lets the
    network learn to reweight channels by importance, image by image.

    'SQUEEZE': global average pooling collapses each channel's entire
    spatial map down to a single number — a compact summary of 'how
    active is this channel, on average, across the whole image'.

    'EXCITE': a tiny two-layer bottleneck network (Conv1x1 -> activation
    -> Conv1x1 -> sigmoid) turns those per-channel summary numbers into
    per-channel importance WEIGHTS between 0 and 1. The bottleneck
    ('reduction') forces the network to compress channel relationships
    into a small representation first, which acts as a form of
    regularization and keeps this block cheap.

    Finally, the original feature map gets multiplied channel-by-channel
    by these learned weights — channels the network decides are useful
    get amplified; channels it decides are noise get suppressed toward
    zero. This is exactly the same core idea used inside EfficientNet.
    """
    def __init__(self, channels, reduction=4):
        super().__init__()
        reduced_channels = max(1, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)                          # squeeze: (B,C,H,W) -> (B,C,1,1)
        self.fc1 = nn.Conv2d(channels, reduced_channels, kernel_size=1)  # bottleneck down
        self.act = nn.SiLU(inplace=True)
        self.fc2 = nn.Conv2d(reduced_channels, channels, kernel_size=1)  # back up to full channel count
        self.gate = nn.Sigmoid()                                      # squashes weights into [0, 1]

    def forward(self, x):
        weights = self.pool(x)
        weights = self.act(self.fc1(weights))
        weights = self.gate(self.fc2(weights))
        return x * weights  # channel-wise reweighting (broadcasts across H, W)

### 2.4 — Residual (Skip) Connections: making deep networks trainable

This is the idea from the 2015 ResNet paper that made it practical to
train much deeper networks than before.

In [ ]:
class ResidualSEBlock(nn.Module):
    """
    Combines a depthwise separable conv + squeeze-excite, then ADDS the
    block's original input back onto its output ('skip connection').

    Why this helps: without a skip connection, each layer has to learn a
    complete transformation from its input to a useful output. WITH a skip
    connection, a layer only has to learn a small CORRECTION on top of
    what it was already given — 'residual learning'. This turns out to be
    a much easier optimization problem in practice, and it also gives
    gradients a direct, short path backward during backpropagation, which
    helps avoid the 'vanishing gradient' problem in deep networks.

    Only used when input and output have the SAME shape (stride=1, no
    channel change) — you can't add two tensors of different shapes
    together, so skip connections are only valid within a stage, not
    across the downsampling steps between stages.
    """
    def __init__(self, channels):
        super().__init__()
        self.conv = DepthwiseSeparableConv(channels, channels, stride=1)
        self.se = SqueezeExcite(channels)

    def forward(self, x):
        out = self.conv(x)
        out = self.se(out)
        return out + x   # <-- the skip connection: add input back onto the transformed output

### 2.5 — Assembling a Stage

A 'stage' groups several blocks together at the same spatial resolution
and channel width, with one downsampling step at the start.

In [ ]:
class DownsampleStage(nn.Module):
    """
    One stage of the network: first, a depthwise separable conv with
    stride=2 HALVES the spatial resolution (224 -> 112 -> 56, etc.) while
    also changing the channel count (e.g. 32 -> 64) -- this first step
    can't use a residual connection since input/output shapes differ.

    After that resolution/channel change, we stack several
    ResidualSEBlocks at the NEW resolution and channel count, where
    residual connections ARE valid (same shape in and out).

    This 'downsample once, then refine several times' pattern is the same
    basic shape used by essentially every modern efficient CNN — early
    stages need more spatial detail but fewer channels; later stages need
    richer channel information but can afford to lose spatial detail.
    """
    def __init__(self, in_channels, out_channels, num_blocks, stride):
        super().__init__()
        layers = [
            DepthwiseSeparableConv(in_channels, out_channels, stride=stride),
            SqueezeExcite(out_channels),
        ]
        for _ in range(num_blocks - 1):
            layers.append(ResidualSEBlock(out_channels))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

### 2.6 — The Full Network: ConductorNet

Now we assemble everything above into the complete model: stem → four
stages → classification head.

In [ ]:
class ConductorNet(nn.Module):
    """
    ConductorNet — an original CNN architecture built specifically for
    this project's per-item visual search task, combining depthwise
    separable convs, squeeze-excitation attention, and residual
    connections into one custom configuration.

    Architecture shape (for width_multiplier=1.0):
      Input (3, 224, 224)
      -> Stem:    3   ->  32 channels,  224 -> 112
      -> Stage1: 32   ->  64 channels,  112 ->  56   (2 blocks)
      -> Stage2: 64   -> 128 channels,   56 ->  28   (3 blocks)
      -> Stage3: 128  -> 256 channels,   28 ->  14   (4 blocks)
      -> Stage4: 256  -> 384 channels,   14 ->   7   (2 blocks)
      -> Global average pool -> (384,) vector
      -> Dropout -> Linear -> num_classes logits

    The 'narrow-and-fine, then wide-and-coarse' progression (few channels
    at high resolution, many channels at low resolution) mirrors what
    every modern efficient CNN does, for the same underlying reason: the
    total amount of COMPUTE per stage (channels x spatial_size) stays
    roughly balanced across the network rather than blowing up.
    """
    def __init__(self, num_classes, width_multiplier=1.0):
        super().__init__()

        def ch(c):
            # Scales every channel count by width_multiplier, rounded to the
            # nearest multiple of 8 -- a common practice since GPUs process
            # channel counts in these chunks more efficiently.
            return int(round(c * width_multiplier / 8) * 8)

        self.stem = ConvBlock(3, ch(32), kernel_size=3, stride=2)

        self.stage1 = DownsampleStage(ch(32), ch(64), num_blocks=2, stride=2)
        self.stage2 = DownsampleStage(ch(64), ch(128), num_blocks=3, stride=2)
        self.stage3 = DownsampleStage(ch(128), ch(256), num_blocks=4, stride=2)
        self.stage4 = DownsampleStage(ch(256), ch(384), num_blocks=2, stride=2)

        self.pool = nn.AdaptiveAvgPool2d(1)   # (B, C, H, W) -> (B, C, 1, 1): a compact summary vector per image
        self.dropout = nn.Dropout(p=0.3)       # randomly zeroes 30% of activations during training only --
                                                 # a crucial regularizer here given how little data per class we have;
                                                 # without it, a from-scratch model would badly overfit even faster
        self.classifier = nn.Linear(ch(384), num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        """
        Weight initialization matters enormously for training FROM SCRATCH
        (unlike transfer learning, where pretrained weights already start
        from a good point). Kaiming/He initialization is specifically
        designed for layers followed by ReLU-family activations (SiLU
        included) — it sets initial weight variance so that signal
        neither shrinks toward zero nor explodes as it passes through many
        stacked layers, which is exactly the failure mode that made very
        deep networks nearly untrainable before this technique existed.
        """
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)   # (B, C, 1, 1) -> (B, C)
        x = self.dropout(x)
        x = self.classifier(x)
        return x   # raw logits — softmax is applied later, not here (CrossEntropyLoss expects raw logits)

## 3. Sanity-Check the Architecture Before Training

In [ ]:
model_check = ConductorNet(num_classes=NUM_CLASSES, width_multiplier=WIDTH_MULTIPLIER)
total_params = sum(p.numel() for p in model_check.parameters())
print(f"ConductorNet parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"For comparison — EfficientNet-B0: 5.07M | MobileNetV2: 3.28M | CLIP: 151.3M")

# Forward-pass shape check with a dummy batch — catches shape bugs before
# wasting time on a full training run
dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
dummy_output = model_check(dummy_input)
print(f"\nDummy input shape:  {dummy_input.shape}")
print(f"Dummy output shape: {dummy_output.shape}")
assert dummy_output.shape == (2, NUM_CLASSES), 'Shape mismatch — architecture has a bug!'
print('✅ Shape check passed.')
del model_check, dummy_input, dummy_output

## 4. Training Loop — Same Early Stopping / Checkpoint Approach as Before

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def model_size_mb(model):
    tmp_path = '_tmp_size_check.pt'
    torch.save(model.state_dict(), tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
    os.remove(tmp_path)
    return size_mb

def measure_inference_speed(model, loader, n_batches=5):
    model.eval()
    times = []
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            if i >= n_batches:
                break
            images = images.to(DEVICE)
            start = time.time()
            _ = model(images)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            times.append((time.time() - start) / images.size(0))
    return float(np.mean(times)) * 1000 if times else float('nan')

In [ ]:
def train_conductornet():
    print(f"\n{'='*60}\nTraining: ConductorNet (from scratch, no pretraining)\n{'='*60}")

    model = ConductorNet(NUM_CLASSES, width_multiplier=WIDTH_MULTIPLIER).to(DEVICE)
    total_params, trainable_params = count_params(model)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    # Cosine learning rate schedule: gradually decays LR over training,
    # which tends to help from-scratch models settle into a better final
    # minimum than a constant learning rate would.
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}
    best_val_acc, best_epoch, best_state = -1.0, -1, None
    epochs_without_improvement = 0

    start_time = time.time()
    for epoch in range(MAX_EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
        train_loss, train_acc = running_loss / total, correct / total
        scheduler.step()

        model.eval()
        val_loss_total, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss_total += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += images.size(0)
        val_loss, val_acc = val_loss_total / val_total, val_correct / val_total

        history['train_acc'].append(train_acc); history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc); history['val_loss'].append(val_loss)

        improved = val_acc > best_val_acc
        print(f"  Epoch {epoch+1}/{MAX_EPOCHS} — train_acc: {train_acc:.3f} train_loss: {train_loss:.3f} "
              f"| val_acc: {val_acc:.3f} val_loss: {val_loss:.3f}{' *' if improved else ''}")

        if improved:
            best_val_acc, best_epoch = val_acc, epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"  ⏹ Early stopping — best was epoch {best_epoch} (val_acc {best_val_acc:.3f}).")
                break

    train_time_sec = time.time() - start_time
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  Restored best checkpoint from epoch {best_epoch}.")

    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    test_loss_total = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images_dev, labels_dev = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images_dev)
            loss = criterion(outputs, labels_dev)
            test_loss_total += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    test_loss = test_loss_total / len(all_labels)
    test_acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
    all_probs_arr = np.array(all_probs)
    k = min(3, NUM_CLASSES)
    top3_acc = top_k_accuracy_score(all_labels, all_probs_arr, k=k, labels=list(range(NUM_CLASSES)))
    try:
        roc_auc = roc_auc_score(all_labels, all_probs_arr, multi_class='ovr', average='macro')
    except Exception:
        roc_auc = float('nan')

    size_mb = model_size_mb(model)
    inference_ms = measure_inference_speed(model, test_loader)

    result = {
        'model_name': 'ConductorNet (custom, from scratch)',
        'total_params': total_params, 'trainable_params': trainable_params,
        'model_size_mb': size_mb, 'train_time_sec': train_time_sec,
        'inference_ms_per_image': inference_ms, 'best_epoch': best_epoch, 'epochs_run': len(history['train_acc']),
        'train_acc': history['train_acc'][best_epoch-1], 'train_loss': history['train_loss'][best_epoch-1],
        'val_acc': best_val_acc, 'val_loss': history['val_loss'][best_epoch-1],
        'test_acc': test_acc, 'test_loss': test_loss,
        'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1,
        'top3_acc': top3_acc, 'roc_auc_macro': roc_auc,
    }
    return result, history

## 5. Train ConductorNet

In [ ]:
conductornet_result, conductornet_history = train_conductornet()
print('\n✅ ConductorNet training complete.')

## 6. Training Curves

In [ ]:
epochs_range = range(1, len(conductornet_history['train_acc']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs_range, conductornet_history['train_acc'], label='Train Acc', marker='o', markersize=3)
axes[0].plot(epochs_range, conductornet_history['val_acc'], label='Val Acc', marker='s', markersize=3)
axes[0].axvline(conductornet_result['best_epoch'], color='green', linestyle='--', alpha=0.6,
                label=f"Best epoch ({conductornet_result['best_epoch']})")
axes[0].set_title('ConductorNet — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=9)

axes[1].plot(epochs_range, conductornet_history['train_loss'], label='Train Loss', marker='o', markersize=3)
axes[1].plot(epochs_range, conductornet_history['val_loss'], label='Val Loss', marker='s', markersize=3)
axes[1].axvline(conductornet_result['best_epoch'], color='green', linestyle='--', alpha=0.6,
                label=f"Best epoch ({conductornet_result['best_epoch']})")
axes[1].set_title('ConductorNet — Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

gap = conductornet_history['train_acc'][conductornet_result['best_epoch']-1] - conductornet_result['val_acc']
print(f"Train/val accuracy gap at best epoch: {gap:.3f} — a large gap here would indicate")
print("overfitting, which is the expected risk when training from scratch on this little data.")

## 7. Final Comparison — ConductorNet vs. the Transfer-Learned Models

Using the results from the previous notebook for the pretrained models,
compared directly against ConductorNet's results from this run.

In [ ]:
prior_results = [
    {'model_name': 'EfficientNet-B0 (pretrained)', 'test_acc': 0.9746, 'top3_acc': 0.9976,
     'f1_macro': 0.9670, 'model_size_mb': 19.6162, 'inference_ms_per_image': 1.3306, 'total_params': 5066935},
    {'model_name': 'MobileNetV2 (pretrained)', 'test_acc': 0.9686, 'top3_acc': 0.9976,
     'f1_macro': 0.9592, 'model_size_mb': 12.7604, 'inference_ms_per_image': 0.8473, 'total_params': 3283259},
    {'model_name': 'CLIP ViT-B/32 (frozen, linear probe)', 'test_acc': 0.9528, 'top3_acc': 0.9964,
     'f1_macro': 0.9386, 'model_size_mb': 577.2275, 'inference_ms_per_image': 7.9913, 'total_params': 151277313},
]

comparison_rows = prior_results + [{
    'model_name': conductornet_result['model_name'],
    'test_acc': conductornet_result['test_acc'],
    'top3_acc': conductornet_result['top3_acc'],
    'f1_macro': conductornet_result['f1_macro'],
    'model_size_mb': conductornet_result['model_size_mb'],
    'inference_ms_per_image': conductornet_result['inference_ms_per_image'],
    'total_params': conductornet_result['total_params'],
}]

comparison_df = pd.DataFrame(comparison_rows).sort_values('test_acc', ascending=False).reset_index(drop=True)
comparison_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#C44E52' if 'ConductorNet' in n else '#4C72B0' for n in comparison_df['model_name']]
axes[0].barh(comparison_df['model_name'], comparison_df['test_acc'], color=colors)
axes[0].set_title('Test Accuracy — ConductorNet (red) vs. Transfer Learning (blue)')
axes[0].set_xlabel('Test Accuracy')

axes[1].scatter(comparison_df['model_size_mb'], comparison_df['test_acc'],
                 s=150, c=colors)
for _, row in comparison_df.iterrows():
    axes[1].annotate(row['model_name'].split(' (')[0], (row['model_size_mb'], row['test_acc']),
                      fontsize=8, xytext=(5,5), textcoords='offset points')
axes[1].set_title('Accuracy vs. Model Size')
axes[1].set_xlabel('Model Size (MB)')
axes[1].set_ylabel('Test Accuracy')

plt.tight_layout()
plt.show()

## 8. Conclusion — What This Experiment Actually Shows

In [ ]:
gap_to_best = comparison_df.iloc[0]['test_acc'] - conductornet_result['test_acc']
conductornet_rank = comparison_df.reset_index()[comparison_df.reset_index()['model_name'] == conductornet_result['model_name']].index[0] + 1

print('='*70)
print('CONDUCTORNET — RESULTS SUMMARY')
print('='*70)
print(f"ConductorNet test accuracy:  {conductornet_result['test_acc']:.3f}")
print(f"Best model (transfer learning): {comparison_df.iloc[0]['model_name']} at {comparison_df.iloc[0]['test_acc']:.3f}")
print(f"Gap: {gap_to_best:.3f} ({gap_to_best*100:.1f} percentage points)")
print(f"ConductorNet ranked #{conductornet_rank} out of {len(comparison_df)}")
print(f"ConductorNet parameter count: {conductornet_result['total_params']:,} "
      f"(smallest of all 4 models, if true)")
print()
print('WHAT THIS DOES AND DOES NOT PROVE:')
print('- If ConductorNet underperformed (expected): this confirms the well-established')
print('  principle that transfer learning from a large pretrained dataset (ImageNet)')
print('  beats training from scratch when your own dataset is small — a real, useful')
print('  lesson, not a failure of the custom architecture design itself.')
print('- If ConductorNet came close or matched: that would be a genuinely interesting')
print('  result worth investigating further — check the train/val gap in Section 6')
print('  first to rule out lucky overfitting to this particular tiny test set')
print('  (remember: often just 1 real test image per class).')
print('- Either way, you now understand — from first principles, not just as an API')
print('  call — what a Conv layer, BatchNorm, depthwise separable convolutions,')
print('  squeeze-excitation attention, and residual connections each actually do,')
print('  and why real architectures like EfficientNet are built the way they are.')